Imports

In [28]:
import pandas as pd

from sklearn.metrics import  root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.feature_selection import  mutual_info_regression, SelectPercentile

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import  OrdinalEncoder
from sklearn.linear_model import RANSACRegressor, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor , HistGradientBoostingRegressor
from sklearn.svm import SVR

import matplotlib.pyplot as plt

Ouverture des données


In [29]:

file_path = "BDD_nettoyé_IA.csv"

data=pd.read_csv(file_path,encoding="utf8",sep=",",low_memory=False)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135008 entries, 0 to 135007
Data columns (total 52 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   nom_amenageur                        134610 non-null  object 
 1   siren_amenageur                      67896 non-null   float64
 2   contact_amenageur                    74216 non-null   object 
 3   nom_operateur                        130843 non-null  object 
 4   contact_operateur                    135008 non-null  object 
 5   telephone_operateur                  85855 non-null   object 
 6   nom_enseigne                         135008 non-null  object 
 7   id_station_itinerance                135008 non-null  object 
 8   id_station_local                     83482 non-null   object 
 9   nom_station                          135008 non-null  object 
 10  implantation_station                 135008 non-null  object 
 11  adresse_stati

Pour traiter les NA


In [30]:

data = data.dropna()


Sélection des données d'apprentissage


In [31]:

dataY = data["puissance_nominale"].copy()

dataX= data.drop(columns="puissance_nominale")


Pour mettre les données textuelles au format numérique

In [32]:
encoder= OrdinalEncoder()

print(dataX.keys())

for index in dataX.keys():
    dataX[[index]]= encoder.fit_transform(dataX[[index]])

dataX= pd.DataFrame(dataX)

Index(['nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'prise_type_ef', 'prise_type_2', 'prise_type_combo_ccs',
       'prise_type_chademo', 'prise_type_autre', 'gratuit', 'paiement_acte',
       'paiement_cb', 'paiement_autre', 'tarification', 'condition_acces',
       'reservation', 'horaires', 'accessibilite_pmr', 'restriction_gabarit',
       'station_deux_roues', 'raccordement', 'num_pdl', 'date_mise_en_service',
       'observations', 'date_maj', 'cable_t2_attache', 'last_modified',
       'datagouv_dataset_id', 'datagouv_resource_id',
       'datagouv_organization_or_owner', 'created_at',
       'consolidated_longitude', 'consolidated_latitude',
       'consolidated_code_p

Choix des colonnes à utiliser par feature selection avec le score de gain d'information

In [33]:

print(dataX.keys())

selection= SelectPercentile(mutual_info_regression,percentile=30)

selectedX = selection.fit_transform(dataX,dataY)


Index(['nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'prise_type_ef', 'prise_type_2', 'prise_type_combo_ccs',
       'prise_type_chademo', 'prise_type_autre', 'gratuit', 'paiement_acte',
       'paiement_cb', 'paiement_autre', 'tarification', 'condition_acces',
       'reservation', 'horaires', 'accessibilite_pmr', 'restriction_gabarit',
       'station_deux_roues', 'raccordement', 'num_pdl', 'date_mise_en_service',
       'observations', 'date_maj', 'cable_t2_attache', 'last_modified',
       'datagouv_dataset_id', 'datagouv_resource_id',
       'datagouv_organization_or_owner', 'created_at',
       'consolidated_longitude', 'consolidated_latitude',
       'consolidated_code_p

Découpage en bases d'apprentissage, de validation et de test

In [34]:

Xtrain, Xtest = train_test_split(selectedX,train_size=0.7)

ytrain, ytest = train_test_split(dataY,train_size=0.7)


Standard scaler pour le SVR

In [35]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

Xtrain_sc = scaler_X.fit_transform(Xtrain)
Xtest_sc = scaler_X.transform(Xtest)

ytrain_sc = scaler_y.fit_transform(ytrain.values.reshape(-1, 1)).ravel()
ytest_sc = scaler_y.transform(ytest.values.reshape(-1, 1)).ravel()

Mise en place et première évaluation des modèles de régression

In [36]:

forest= RandomForestRegressor(random_state=42, oob_score=True,bootstrap=True)

forest.fit(X=Xtrain,y=ytrain)

ypred_forest= forest.predict(X=Xtest)

print("score RMSE random forest:",root_mean_squared_error(y_pred=ypred_forest,y_true=ytest))
print("score MAE random forest:",mean_absolute_error(y_pred=ypred_forest,y_true=ytest))
print("score R² random forest:",r2_score(y_pred=ypred_forest,y_true=ytest))
print("score OOB:",forest.oob_score_)

svr= SVR()

svr.fit(X=Xtrain_sc,y=ytrain_sc)

ypred_svr= svr.predict(X=Xtest_sc)

ypred_svr= scaler_y.inverse_transform(ypred_svr.reshape(-1, 1))

print("score RMSE svr:",root_mean_squared_error(y_pred=ypred_svr,y_true=ytest_sc))
print("score R² random svr:",r2_score(y_pred=ypred_svr,y_true=ytest))
print("score MAE random svr:",mean_absolute_error(y_pred=ypred_svr,y_true=ytest))

ransac= RANSACRegressor(estimator=LinearRegression())

ransac.fit(X=Xtrain,y=ytrain)

ypred_ransac= ransac.predict(X=Xtest)

print("score RMSE ransac:",root_mean_squared_error(y_pred=ypred_ransac,y_true=ytest))
print("score R² ransac:",r2_score(y_pred=ypred_ransac,y_true=ytest))
print("score MAE ransac:",mean_absolute_error(y_pred=ypred_ransac,y_true=ytest))

boost= HistGradientBoostingRegressor()

boost.fit(X=Xtrain,y=ytrain)

ypred_boost= boost.predict(X=Xtest)

print("score RMSE boosting:",root_mean_squared_error(y_pred=ypred_boost,y_true=ytest))
print("score R² boosting:",r2_score(y_pred=ypred_boost,y_true=ytest))
print("score MAE boosting:",mean_absolute_error(y_pred=ypred_boost,y_true=ytest))



score RMSE random forest: 169.57714418096393
score MAE random forest: 133.30378571428568
score R² random forest: -0.2119333317278167
score OOB: -0.5804824121184824
score RMSE svr: 327.11741245897423
score R² random svr: -0.2783999886804076
score MAE random svr: 112.21406398143134


C:\Users\Utilisateur\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
C:\Users\Utilisateur\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
C:\Users\Utilisateur\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_regression.py:1288: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


score RMSE ransac: 186.9858665399989
score R² ransac: -0.4735390802798416
score MAE ransac: 106.0
score RMSE boosting: 153.3704605600894
score R² boosting: 0.008648794014849526
score MAE boosting: 133.79939061878827


utilisation de la search grid pour déterminer les meilleurs paramètres

In [37]:
param_grid_forest = {
    'max_features': [1.0, 'sqrt', 'log2'],
    'max_depth': [None,3,5,10,15],
    'criterion': ['squared_error', 'absolute_error'],
    'n_estimators': [50,100,200],
    "min_samples_split": [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_forest= GridSearchCV(estimator=forest,param_grid=param_grid_forest,scoring="neg_root_mean_squared_error",cv=KFold(10),n_jobs=-1)


grid_forest.fit(Xtrain, ytrain)

param_grid_svr = {'C': [0.1, 1, 10, 100, 1000],
			'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
			'kernel': ['rbf','linear'],
            'epsilon': [0.01, 0.1, 1],
            }


grid_svr= GridSearchCV(estimator=svr,param_grid=param_grid_svr,scoring="neg_root_mean_squared_error",cv=KFold(10),n_jobs=-1)


grid_svr.fit(Xtrain_sc, ytrain_sc)

param_grid_ransac = {
    'min_samples': [0.1, 0.5, 0.9],
    'residual_threshold': [1.0, 5.0, 10.0],
    'max_trials': [100, 500, 1000]
}

grid_ransac= GridSearchCV(estimator=ransac,param_grid=param_grid_ransac,scoring="neg_root_mean_squared_error",cv=KFold(10),n_jobs=-1)


grid_ransac.fit(Xtrain, ytrain)

param_grid_boost = {
    'learning_rate': [0.01, 0.1, 1],
    'max_iter': [50, 100],
    'max_leaf_nodes': [10, 20, 30]
}

grid_boost= GridSearchCV(estimator=boost,param_grid=param_grid_boost,scoring="neg_root_mean_squared_error",cv=KFold(10),n_jobs=-1)


grid_boost.fit(Xtrain, ytrain)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",HistGradientB...ingRegressor()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.01, 0.1, ...], 'max_iter': [50, 100], 'max_leaf_nodes': [10, 20, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split...shuffle=False)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for e

Résultats

In [38]:
print("--Algorithme RANDOM FOREST--")

print("Best forest parameters found: ", grid_forest.best_params_)
print("Best forest cross-validation score: ", grid_forest.best_score_)
print("Best OOB score: ", grid_forest.best_estimator_.oob_score_)

y_pred_forest = grid_forest.predict(Xtest)

score_forest = root_mean_squared_error(y_true=ytest, y_pred=y_pred_forest)
print("score RMSE forest: ", score_forest)

score_R2_forest = grid_forest.best_estimator_.score(Xtest, ytest)
print("score R² forest:",score_R2_forest)

score_mae_forest = mean_absolute_error(y_true=ytest, y_pred=y_pred_forest)
print("score MAE forest:",score_mae_forest)

print("--Algorithme SVR--")

print("Best svr parameters found: ", grid_svr.best_params_)
print("Best svr cross-validation score: ", grid_svr.best_score_)

y_pred_svr= grid_svr.predict(X=Xtest_sc)

y_pred_svr= scaler_y.inverse_transform(y_pred_svr.reshape(-1, 1))

score_svr = root_mean_squared_error(y_true=ytest, y_pred=y_pred_svr)
print("score RMSE svr: ", score_svr)

score_R2_svr = grid_svr.best_estimator_.score(Xtest_sc, ytest_sc)
print("score R² svr:",score_R2_svr)

score_mae_svr = mean_absolute_error(y_true=ytest, y_pred=y_pred_svr)
print("score MAE svr:",score_mae_svr)

print("--Algorithme RANSAC--")

print("Best ransac parameters found: ", grid_ransac.best_params_)
print("Best ransac cross-validation score: ", grid_ransac.best_score_)

y_pred_ransac= grid_ransac.predict(X=Xtest)

score_ransac = root_mean_squared_error(y_true=ytest, y_pred=y_pred_ransac)
print("score RMSE ransac: ", score_ransac)

score_R2_ransac = grid_ransac.best_estimator_.score(Xtest, ytest)
print("score R² ransac:",score_R2_ransac)

score_mae_ransac = mean_absolute_error(y_true=ytest, y_pred=y_pred_ransac)
print("score MAE ransac:",score_mae_ransac)

print("--Algorithme HISTRANDOMBOOST--")

print("Best boost parameters found: ", grid_boost.best_params_)
print("Best boost cross-validation score: ", grid_boost.best_score_)

y_pred_boost= grid_boost.predict(X=Xtest)

score_boost = root_mean_squared_error(y_true=ytest, y_pred=y_pred_boost)
print("score RMSE boost: ", score_boost)

score_R2_boost = grid_boost.best_estimator_.score(Xtest, ytest)
print("score R² boost",score_R2_ransac)

score_mae_boost = mean_absolute_error(y_true=ytest, y_pred=y_pred_boost)
print("score MAE boost:",score_mae_boost)


--Algorithme RANDOM FOREST--
Best forest parameters found:  {'criterion': 'squared_error', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}
Best forest cross-validation score:  -161.69809789996106
Best OOB score:  -0.05987870479482238
score RMSE forest:  154.12575128118831
score R² forest: -0.0011392973613439938
score MAE forest: 144.14674174435723
--Algorithme SVR--
Best svr parameters found:  {'C': 10, 'epsilon': 1, 'gamma': 0.01, 'kernel': 'rbf'}
Best svr cross-validation score:  -1.0287494242376658
score RMSE svr:  161.8958910758286
score R² svr: -0.10462724018119784
score MAE svr: 161.67667896071154
--Algorithme RANSAC--
Best ransac parameters found:  {'max_trials': 1000, 'min_samples': 0.9, 'residual_threshold': 10.0}
Best ransac cross-validation score:  -188.31786779303724
score RMSE ransac:  208.44015412690942
score R² ransac: -0.8310780941343352
score MAE ransac: 153.93009089447878
--Algorithme HISTRANDOMBOOST--
Best b